# VMamba Semantic Segmentation — Allen Brain Atlas 3D

Two code cells: **cell 1** defines everything (VMamba VSSM backbone, UperNet-lite
decoder, polygon-rasterizing dataset, CE+Dice loss, train/eval helpers).
**Cell 2** runs the training loop — edit the config block at the top.

**Inputs:**
- Images: `S:\\Phys\\FIV911 Atlas\\SimDS\\AllenBA 3D\\20a\\train\\*.bmp` (640×640 RGB)
- Labels: `S:\\Phys\\FIV911 Atlas\\SimDS\\AllenBA 3D\\20a\\semantic\\*.txt` (YOLOv8-seg polygons)

**Dependencies:** `torch`, `numpy`, `opencv-python-headless`. The selective scan
runs in pure PyTorch (sequential); install `mamba_ssm` (Linux/WSL only) for a
~10× speedup — the code auto-detects it.


In [ ]:
"""All VMamba + segmentation training definitions in one cell."""
import json, math, os, random, sys, time
from pathlib import Path
from typing import List, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

IGNORE_INDEX = 255
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


# ---- selective state-space scan ---------------------------------------------

def _selective_scan_torch(u, delta, A, B, C, D):
    """Sequential selective scan (pure PyTorch). Slow but works anywhere."""
    bs, dim, L = u.shape
    N = A.shape[1]
    deltaA = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(2))
    deltaB_u = delta.unsqueeze(-1) * B.permute(0, 2, 1).unsqueeze(1) * u.unsqueeze(-1)
    x = torch.zeros(bs, dim, N, device=u.device, dtype=u.dtype)
    Cp = C.permute(0, 2, 1)
    ys = []
    for i in range(L):
        x = deltaA[:, :, i] * x + deltaB_u[:, :, i]
        ys.append((x * Cp[:, i].unsqueeze(1)).sum(-1))
    return torch.stack(ys, dim=-1) + D.view(1, -1, 1) * u


try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn as _fast_scan
    _HAS_FAST = True
except Exception:
    _fast_scan = None
    _HAS_FAST = False


def _scan(u, delta, A, B, C, D):
    if _HAS_FAST and u.is_cuda:
        return _fast_scan(u, delta, A, B, C, D, z=None, delta_bias=None, delta_softplus=True)
    return _selective_scan_torch(u, F.softplus(delta), A, B, C, D)


# ---- VMamba backbone --------------------------------------------------------

class PatchEmbed(nn.Module):
    def __init__(self, in_chans=3, embed_dim=96):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=4, stride=4)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        return self.norm(self.proj(x).permute(0, 2, 3, 1).contiguous())


class PatchMerging(nn.Module):
    def __init__(self, dim, out_dim):
        super().__init__()
        self.reduction = nn.Linear(4 * dim, out_dim, bias=False)
        self.norm = nn.LayerNorm(4 * dim)
    def forward(self, x):
        B, H, W, C = x.shape
        if H % 2 or W % 2:
            x = F.pad(x, (0, 0, 0, W % 2, 0, H % 2))
            H, W = x.shape[1], x.shape[2]
        x = torch.cat([x[:, 0::2, 0::2], x[:, 1::2, 0::2], x[:, 0::2, 1::2], x[:, 1::2, 1::2]], dim=-1)
        return self.reduction(self.norm(x))


class SS2D(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=3, expand=2.0, dropout=0.0):
        super().__init__()
        self.d_inner = int(d_model * expand)
        self.d_state = d_state
        self.dt_rank = math.ceil(d_model / 16)
        self.K = 4
        D, N, R = self.d_inner, d_state, self.dt_rank
        self.in_proj = nn.Linear(d_model, D * 2, bias=False)
        self.conv2d = nn.Conv2d(D, D, kernel_size=d_conv, padding=d_conv // 2, groups=D, bias=True)
        self.act = nn.SiLU()
        self.x_proj_weight = nn.Parameter(torch.stack(
            [nn.Linear(D, R + 2 * N, bias=False).weight for _ in range(self.K)], dim=0))
        self.dt_proj_weight = nn.Parameter(torch.stack(
            [nn.Linear(R, D, bias=True).weight for _ in range(self.K)], dim=0))
        self.dt_proj_bias = nn.Parameter(torch.zeros(self.K, D))
        A_init = torch.arange(1, N + 1, dtype=torch.float32).unsqueeze(0).expand(self.K * D, -1)
        self.A_logs = nn.Parameter(torch.log(A_init.contiguous()))
        self.Ds = nn.Parameter(torch.ones(self.K * D))
        self.out_norm = nn.LayerNorm(D)
        self.out_proj = nn.Linear(D, d_model, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def _cross_scan(self, x):
        x_h = x.flatten(2)
        x_w = x.transpose(2, 3).flatten(2)
        return torch.stack([x_h, x_w, torch.flip(x_h, dims=[-1]), torch.flip(x_w, dims=[-1])], dim=1)

    def _cross_merge(self, y, H, W):
        y_h, y_w, y_hr, y_wr = y[:, 0], y[:, 1], torch.flip(y[:, 2], dims=[-1]), torch.flip(y[:, 3], dims=[-1])
        out = (y_h + y_hr).view(y.shape[0], -1, H, W)
        out_t = (y_w + y_wr).view(y.shape[0], -1, W, H).transpose(2, 3)
        return out + out_t

    def forward(self, x):
        B, H, W, C = x.shape
        xz = self.in_proj(x)
        x, z = xz.chunk(2, dim=-1)
        x = self.act(self.conv2d(x.permute(0, 3, 1, 2).contiguous()))
        D, N, K = self.d_inner, self.d_state, self.K
        xs = self._cross_scan(x)
        x_dbl = torch.einsum("bkdl,kfd->bkfl", xs, self.x_proj_weight)
        dts, Bs, Cs = torch.split(x_dbl, [self.dt_rank, N, N], dim=2)
        dts = torch.einsum("bkrl,kdr->bkdl", dts, self.dt_proj_weight) + self.dt_proj_bias.view(1, K, D, 1)
        ys_list = []
        for k in range(K):
            A_k = -torch.exp(self.A_logs.view(K, D, N)[k].float())
            y_k = _scan(xs[:, k], dts[:, k], A_k, Bs[:, k], Cs[:, k], self.Ds.view(K, D)[k])
            ys_list.append(y_k)
        y = self._cross_merge(torch.stack(ys_list, dim=1), H, W).permute(0, 2, 3, 1).contiguous()
        return self.dropout(self.out_proj(self.out_norm(y) * F.silu(z)))


class VSSBlock(nn.Module):
    def __init__(self, dim, d_state=16, mlp_ratio=4.0, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.ss2d = SS2D(dim, d_state=d_state)
        self.norm2 = nn.LayerNorm(dim)
        h = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, h), nn.GELU(), nn.Linear(h, dim))
        self.drop_path = drop_path
    def _drop(self, x):
        if self.drop_path == 0.0 or not self.training:
            return x
        keep = 1.0 - self.drop_path
        mask = torch.empty(x.shape[0], 1, 1, 1, device=x.device).bernoulli_(keep).div_(keep)
        return x * mask
    def forward(self, x):
        x = x + self._drop(self.ss2d(self.norm1(x)))
        x = x + self._drop(self.mlp(self.norm2(x)))
        return x


class VSSM(nn.Module):
    def __init__(self, in_chans=3, depths=(2, 2, 4, 2), dims=(96, 192, 384, 768),
                 d_state=16, mlp_ratio=4.0, drop_path_rate=0.1):
        super().__init__()
        depths, dims = list(depths), list(dims)
        self.dims = dims
        self.patch_embed = PatchEmbed(in_chans, dims[0])
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        self.stages = nn.ModuleList()
        self.downs = nn.ModuleList()
        cur = 0
        for i in range(4):
            self.stages.append(nn.ModuleList([
                VSSBlock(dims[i], d_state=d_state, mlp_ratio=mlp_ratio, drop_path=dpr[cur + j])
                for j in range(depths[i])
            ]))
            cur += depths[i]
            if i < 3:
                self.downs.append(PatchMerging(dims[i], dims[i + 1]))
        self.out_norms = nn.ModuleList([nn.LayerNorm(d) for d in dims])
    def forward(self, x):
        feats = []
        x = self.patch_embed(x)
        for i in range(4):
            for blk in self.stages[i]:
                x = blk(x)
            feats.append(self.out_norms[i](x).permute(0, 3, 1, 2).contiguous())
            if i < 3:
                x = self.downs[i](x)
        return feats


# ---- segmentation head ------------------------------------------------------

class PPM(nn.Module):
    def __init__(self, in_dim, out_dim, scales=(1, 2, 3, 6)):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(nn.AdaptiveAvgPool2d(s), nn.Conv2d(in_dim, out_dim, 1, bias=False),
                          nn.BatchNorm2d(out_dim), nn.ReLU(inplace=True))
            for s in scales
        ])
        self.fuse = nn.Sequential(
            nn.Conv2d(in_dim + len(scales) * out_dim, out_dim, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_dim), nn.ReLU(inplace=True))
    def forward(self, x):
        H, W = x.shape[-2:]
        feats = [x] + [F.interpolate(b(x), size=(H, W), mode="bilinear", align_corners=False) for b in self.branches]
        return self.fuse(torch.cat(feats, dim=1))


class UperNetLite(nn.Module):
    def __init__(self, in_dims, decoder_dim=256, num_classes=178):
        super().__init__()
        c1, c2, c3, c4 = in_dims
        self.ppm = PPM(c4, decoder_dim)
        self.laterals = nn.ModuleList([
            nn.Sequential(nn.Conv2d(c, decoder_dim, 1, bias=False), nn.BatchNorm2d(decoder_dim), nn.ReLU(inplace=True))
            for c in (c1, c2, c3)
        ])
        self.fpn_smooth = nn.ModuleList([
            nn.Sequential(nn.Conv2d(decoder_dim, decoder_dim, 3, padding=1, bias=False),
                          nn.BatchNorm2d(decoder_dim), nn.ReLU(inplace=True))
            for _ in range(3)
        ])
        self.fuse = nn.Sequential(
            nn.Conv2d(4 * decoder_dim, decoder_dim, 3, padding=1, bias=False),
            nn.BatchNorm2d(decoder_dim), nn.ReLU(inplace=True), nn.Dropout2d(0.1))
        self.classifier = nn.Conv2d(decoder_dim, num_classes, 1)
    def forward(self, feats):
        c1, c2, c3, c4 = feats
        p4 = self.ppm(c4)
        p3 = self.fpn_smooth[2](self.laterals[2](c3) + F.interpolate(p4, size=c3.shape[-2:], mode="bilinear", align_corners=False))
        p2 = self.fpn_smooth[1](self.laterals[1](c2) + F.interpolate(p3, size=c2.shape[-2:], mode="bilinear", align_corners=False))
        p1 = self.fpn_smooth[0](self.laterals[0](c1) + F.interpolate(p2, size=c1.shape[-2:], mode="bilinear", align_corners=False))
        ts = c1.shape[-2:]
        ps = [p1] + [F.interpolate(p, size=ts, mode="bilinear", align_corners=False) for p in (p2, p3, p4)]
        return self.classifier(self.fuse(torch.cat(ps, dim=1)))


class VMambaSegmenter(nn.Module):
    def __init__(self, num_classes=178, depths=(2, 2, 4, 2), dims=(96, 192, 384, 768),
                 decoder_dim=256, drop_path_rate=0.1):
        super().__init__()
        self.backbone = VSSM(depths=depths, dims=dims, drop_path_rate=drop_path_rate)
        self.head = UperNetLite(list(dims), decoder_dim=decoder_dim, num_classes=num_classes)
    def forward(self, x):
        H, W = x.shape[-2:]
        feats = self.backbone(x)
        return F.interpolate(self.head(feats), size=(H, W), mode="bilinear", align_corners=False)


def build_segmenter(variant="tiny", num_classes=178):
    cfg = {
        "mini":  dict(depths=(1, 1, 2, 1), dims=(48, 96, 192, 384)),
        "tiny":  dict(depths=(2, 2, 4, 2), dims=(96, 192, 384, 768)),
        "small": dict(depths=(2, 2, 9, 2), dims=(96, 192, 384, 768)),
    }[variant]
    return VMambaSegmenter(num_classes=num_classes, **cfg)


# ---- dataset (rasterizes YOLOv8-seg polygons on the fly) --------------------

class AllenBASegDataset(Dataset):
    def __init__(self, img_dir, label_dir, size=640, augment=False, img_suffix=".bmp"):
        self.img_dir = Path(img_dir); self.label_dir = Path(label_dir)
        self.size = size; self.augment = augment
        self.samples = []
        for lbl in sorted(self.label_dir.glob("*.txt")):
            img = self.img_dir / (lbl.stem + img_suffix)
            if img.exists():
                self.samples.append((img, lbl))
        if not self.samples:
            raise RuntimeError(f"No pairs found: img_dir={self.img_dir} label_dir={self.label_dir}")
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(str(img_path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        if img.shape[0] != self.size or img.shape[1] != self.size:
            img = cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_LINEAR)
        mask = np.full((self.size, self.size), IGNORE_INDEX, dtype=np.uint8)
        with lbl_path.open("r", encoding="utf-8", errors="replace") as fh:
            for line in fh:
                parts = line.split()
                if len(parts) < 7:
                    continue
                try:
                    cid = int(parts[0])
                except ValueError:
                    continue
                coords = np.asarray(parts[1:], dtype=np.float32)
                if coords.size % 2 != 0:
                    continue
                px = np.clip(np.round(coords.reshape(-1, 2) * self.size).astype(np.int32), 0, self.size - 1)
                cv2.fillPoly(mask, [px.reshape(-1, 1, 2)], color=cid)
        if self.augment:
            if random.random() < 0.5:
                img = np.ascontiguousarray(img[:, ::-1, :]); mask = np.ascontiguousarray(mask[:, ::-1])
            if random.random() < 0.5:
                img = np.ascontiguousarray(img[::-1, :, :]); mask = np.ascontiguousarray(mask[::-1, :])
        img = (img.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
        return torch.from_numpy(img.transpose(2, 0, 1)).float(), torch.from_numpy(mask.astype(np.int64))


# ---- losses, metrics, train/eval loops --------------------------------------

class CEDiceLoss(nn.Module):
    def __init__(self, num_classes, ignore_index=IGNORE_INDEX, dice_weight=0.5):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.dice_weight = dice_weight
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index)
    def forward(self, logits, target):
        ce_loss = self.ce(logits, target)
        valid = target != self.ignore_index
        if valid.sum() == 0:
            return ce_loss
        probs = F.softmax(logits, dim=1)
        tgt = target.clone(); tgt[~valid] = 0
        oh = F.one_hot(tgt, num_classes=self.num_classes).permute(0, 3, 1, 2).float()
        v = valid.unsqueeze(1).float()
        oh = oh * v; probs = probs * v
        dims = (0, 2, 3)
        inter = (probs * oh).sum(dims)
        denom = probs.sum(dims) + oh.sum(dims) + 1e-6
        present = oh.sum(dims) > 0
        dice = (2 * inter[present] / denom[present]).mean() if present.any() else logits.new_zeros(())
        return ce_loss + self.dice_weight * (1.0 - dice)


@torch.no_grad()
def update_confusion(cm, logits, target, ignore_index):
    pred = logits.argmax(dim=1)
    valid = target != ignore_index
    t = target[valid].cpu().numpy()
    p = pred[valid].cpu().numpy()
    n = cm.shape[0]
    cm += np.bincount(t * n + p, minlength=n * n).reshape(n, n)


def cm_metrics(cm):
    inter = np.diag(cm); pred_sum = cm.sum(0); gt_sum = cm.sum(1)
    union = pred_sum + gt_sum - inter; present = gt_sum > 0
    iou = np.where(union > 0, inter / np.maximum(union, 1), 0.0)
    return {
        "miou": float(iou[present].mean()) if present.any() else 0.0,
        "pixel_acc": float(inter.sum() / max(cm.sum(), 1)),
    }


def train_one_epoch(model, loader, optimizer, scaler, loss_fn, device, log_every=20):
    model.train()
    running, n, t0 = 0.0, 0, time.time()
    for it, (img, mask) in enumerate(loader, 1):
        img = img.to(device, non_blocking=True); mask = mask.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                logits = model(img); loss = loss_fn(logits, mask)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(img); loss = loss_fn(logits, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        running += float(loss.item()) * img.size(0); n += img.size(0)
        if it % log_every == 0:
            dt = time.time() - t0
            print(f"  it {it}/{len(loader)}  loss={running / max(n, 1):.4f}  {n / dt:.1f} img/s")
    return running / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, loss_fn, device, num_classes):
    model.eval()
    running, n = 0.0, 0
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for img, mask in loader:
        img = img.to(device, non_blocking=True); mask = mask.to(device, non_blocking=True)
        logits = model(img); loss = loss_fn(logits, mask)
        running += float(loss.item()) * img.size(0); n += img.size(0)
        update_confusion(cm, logits, mask, IGNORE_INDEX)
    m = cm_metrics(cm); m["loss"] = running / max(n, 1)
    return m


def cosine_lr(base_lr, warmup_iters, total_iters):
    def lr_at(it):
        if it < warmup_iters:
            return base_lr * (it + 1) / max(1, warmup_iters)
        prog = (it - warmup_iters) / max(1, total_iters - warmup_iters)
        return base_lr * 0.5 * (1 + math.cos(math.pi * prog))
    return lr_at


print(f"definitions loaded; mamba_ssm fast path: {_HAS_FAST}")
print(f"torch: {torch.__version__}  cuda: {torch.cuda.is_available()}",
      f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "")


In [ ]:
# ============================================================================
# CONFIG  — edit these
# ============================================================================
IMG_DIR     = r"S:\Phys\FIV911 Atlas\SimDS\AllenBA 3D\20a\train"
LABEL_DIR   = r"S:\Phys\FIV911 Atlas\SimDS\AllenBA 3D\20a\semantic"
OUT_DIR     = r"C:\Users\FIVE\source\repos\Jess\vmamba_runs"

VARIANT     = "tiny"   # "mini" (~12M), "tiny" (~35M), "small" (~50M)
NUM_CLASSES = 178
SIZE        = 384      # input square size; 640 is native but tight on 8GB VRAM
BATCH_SIZE  = 2
EPOCHS      = 100
LR          = 1e-4
WEIGHT_DECAY= 0.05
NUM_WORKERS = 4
VAL_FRAC    = 0.05
SEED        = 0
USE_AMP     = True
SMOKE_TEST  = False    # True = 1 epoch on 2 samples, no checkpoint write
RESUME      = ""       # path to last.pt to resume from
# ============================================================================

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}  amp: {USE_AMP and device.type == 'cuda'}")
if device.type == "cuda":
    print(f"  cuda: {torch.cuda.get_device_name(0)}")

full = AllenBASegDataset(IMG_DIR, LABEL_DIR, size=SIZE, augment=True)
print(f"dataset: {len(full)} samples  size={SIZE}")

rng = random.Random(SEED)
idx = list(range(len(full))); rng.shuffle(idx)
n_val = max(1, int(round(len(full) * VAL_FRAC)))
train_idx, val_idx = idx[n_val:], idx[:n_val]
train_ds = Subset(full, train_idx)
val_full = AllenBASegDataset(IMG_DIR, LABEL_DIR, size=SIZE, augment=False)
val_ds = Subset(val_full, val_idx)

if SMOKE_TEST:
    train_ds = Subset(full, train_idx[:2])
    val_ds   = Subset(val_full, val_idx[:2])
    EPOCHS = 1

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=device.type=="cuda",
                          drop_last=True, persistent_workers=NUM_WORKERS > 0)
val_loader   = DataLoader(val_ds,   batch_size=max(1, BATCH_SIZE), shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=device.type=="cuda",
                          persistent_workers=NUM_WORKERS > 0)

model = build_segmenter(VARIANT, num_classes=NUM_CLASSES).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"model: VMamba-{VARIANT}  params: {n_params:.1f}M  num_classes: {NUM_CLASSES}")

loss_fn   = CEDiceLoss(num_classes=NUM_CLASSES).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = torch.amp.GradScaler("cuda") if (USE_AMP and device.type == "cuda") else None

total_iters  = max(1, EPOCHS * len(train_loader))
warmup_iters = min(500, max(1, total_iters // 20))
lr_fn        = cosine_lr(LR, warmup_iters, total_iters)

out = Path(OUT_DIR); out.mkdir(parents=True, exist_ok=True)
log_path = out / "log.jsonl"

start_epoch, best_miou = 0, -1.0
if RESUME and Path(RESUME).is_file():
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt.get("epoch", 0) + 1
    best_miou   = ckpt.get("best_miou", -1.0)
    print(f"resumed from {RESUME}  epoch={start_epoch}  best_miou={best_miou:.4f}")

global_it = start_epoch * len(train_loader)
for epoch in range(start_epoch, EPOCHS):
    print(f"\n=== epoch {epoch + 1}/{EPOCHS} ===")
    for g in optimizer.param_groups:
        g["lr"] = lr_fn(global_it)
    train_loss = train_one_epoch(model, train_loader, optimizer, scaler, loss_fn, device)
    global_it += len(train_loader)
    val_m   = evaluate(model, val_loader, loss_fn, device, NUM_CLASSES)
    cur_lr  = optimizer.param_groups[0]["lr"]
    print(f"epoch {epoch + 1}  train_loss={train_loss:.4f}  val_loss={val_m['loss']:.4f}  "
          f"val_miou={val_m['miou']:.4f}  val_acc={val_m['pixel_acc']:.4f}  lr={cur_lr:.2e}")
    with log_path.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps({"epoch": epoch + 1, "train_loss": train_loss,
                             "val_loss": val_m["loss"], "val_miou": val_m["miou"],
                             "val_pixel_acc": val_m["pixel_acc"], "lr": cur_lr}) + "\n")
    if SMOKE_TEST:
        print("smoke-test complete"); break
    ckpt = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "epoch": epoch, "best_miou": max(best_miou, val_m["miou"])}
    torch.save(ckpt, out / "last.pt")
    if val_m["miou"] > best_miou:
        best_miou = val_m["miou"]
        torch.save(ckpt, out / "best.pt")
        print(f"  saved best.pt  miou={best_miou:.4f}")

print(f"\ndone. best_val_miou={best_miou:.4f}")
